In [ ]:
import pandas as pd

# 1. 열 이름 읽기 (Windows 줄바꿈 \r\n 대비 strip 처리)
with open("Faults27x7_var", "r") as f:
    cols = [line.strip() for line in f if line.strip()]

# 2. 데이터 읽기 (탭/공백 구분, 헤더 없음)
data = pd.read_csv("Faults.NNA", sep=r"\s+", header=None, names=cols)

# 3. 확인
print(data.shape)          # (1941, 34)
print(data.isna().sum().sum())  # 0

# 4. CSV로 저장
data.to_csv("steel_plates_faults.csv", index=False, encoding="utf-8-sig")

df = pd.read_csv("steel_plates_faults.csv")
display(df.head())
display(df.shape)         
display(df.isna().sum().sum())
display(df.info())

(1941, 34)
0


,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,...,Orientation_Index,Luminosity_Index,SigmoidOfAreas,Pastry,Z_Scratch,K_Scatch,Stains,Dirtiness,Bumps,Other_Faults
0,42,50,270900,270944,267,17,44,24220,76,108,...,0.8182,-0.2913,0.5822,1,0,0,0,0,0,0
1,645,651,2538079,2538108,108,10,30,11397,84,123,...,0.7931,-0.1756,0.2984,1,0,0,0,0,0,0
2,829,835,1553913,1553931,71,8,19,7972,99,125,...,0.6667,-0.1228,0.2150,1,0,0,0,0,0,0
3,853,860,369370,369415,176,13,45,18996,99,126,...,0.8444,-0.1568,0.5212,1,0,0,0,0,0,0
4,1289,1306,498078,498335,2409,60,260,246930,37,126,...,0.9338,-0.1992,1.0000,1,0,0,0,0,0,0


(1941, 34)

np.int64(0)

<class 'pandas.DataFrame'>
RangeIndex: 1941 entries, 0 to 1940
Data columns (total 34 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   X_Minimum              1941 non-null   int64  
 1   X_Maximum              1941 non-null   int64  
 2   Y_Minimum              1941 non-null   int64  
 3   Y_Maximum              1941 non-null   int64  
 4   Pixels_Areas           1941 non-null   int64  
 5   X_Perimeter            1941 non-null   int64  
 6   Y_Perimeter            1941 non-null   int64  
 7   Sum_of_Luminosity      1941 non-null   int64  
 8   Minimum_of_Luminosity  1941 non-null   int64  
 9   Maximum_of_Luminosity  1941 non-null   int64  
 10  Length_of_Conveyer     1941 non-null   int64  
 11  TypeOfSteel_A300       1941 non-null   int64  
 12  TypeOfSteel_A400       1941 non-null   int64  
 13  Steel_Plate_Thickness  1941 non-null   int64  
 14  Edges_Index            1941 non-null   float64
 15  Empty_Index    

None

TypeOfSteel
1    1164
0     777
Name: count, dtype: int64

# 전처리

In [11]:
# 파생 변수 확인

"""
강판 결함 데이터: 지수(Index) 변수가 기본 측정값으로 계산된 파생 변수인지 검증
- 계산식으로 직접 값을 만들고, 원본 열과 비교해 일치율과 오차를 출력합니다.
- 원본 값은 소수점 4자리로 반올림되어 있으므로 허용 오차(TOL) 이내면 일치로 봅니다.
"""
import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# 1. 데이터 불러오기 (CSV로 변환한 파일 사용)
# ---------------------------------------------------------------
df = pd.read_csv("steel_plates_faults.csv")
# NNA 원본을 바로 쓰려면 아래 두 줄로 대체
# cols = [c.strip() for c in open("Faults27x7_var") if c.strip()]
# df = pd.read_csv("Faults.NNA", sep=r"\s+", header=None, names=cols)

TOL = 1e-3  # 반올림 오차 허용 범위

# ---------------------------------------------------------------
# 2. 계산에 쓸 기본 값
# ---------------------------------------------------------------
dx = df["X_Maximum"] - df["X_Minimum"]          # 결함 가로 길이
dy = df["Y_Maximum"] - df["Y_Minimum"]          # 결함 세로 길이
L = df["Length_of_Conveyer"]                    # 컨베이어(강판 폭) 길이
area = df["Pixels_Areas"]                       # 결함 면적
mean_lum = df["Sum_of_Luminosity"] / area       # 평균 밝기

# ---------------------------------------------------------------
# 3. 계산식으로 파생 변수 직접 생성
# ---------------------------------------------------------------
calc = pd.DataFrame(index=df.index)
calc["Edges_Index"] = 2 * np.minimum(df["X_Minimum"], L - df["X_Maximum"]) / L
calc["Empty_Index"] = 1 - area / (dx * dy)
calc["Square_Index"] = np.minimum(dx, dy) / np.maximum(dx, dy)
calc["Outside_X_Index"] = dx / L
calc["Edges_X_Index"] = dx / df["X_Perimeter"]
calc["Edges_Y_Index"] = dy / df["Y_Perimeter"]
calc["Outside_Global_Index"] = np.select([dy > dx, dy == dx], [1.0, 0.5], default=0.0)
calc["LogOfAreas"] = np.log10(area)
calc["Log_X_Index"] = np.log10(dx)
calc["Log_Y_Index"] = np.log10(dy)
calc["Orientation_Index"] = np.where(dy >= dx, 1 - dx / dy, -(1 - dy / dx))
calc["Luminosity_Index"] = mean_lum / 128 - 1

# ---------------------------------------------------------------
# 4. 원본 값과 비교
# ---------------------------------------------------------------
rows = []
for col in calc.columns:
    diff = (df[col] - calc[col]).abs()
    rows.append({
        "변수": col,
        "일치 행 수": int((diff <= TOL).sum()),
        "전체 행 수": len(df),
        "일치율(%)": round((diff <= TOL).mean() * 100, 2),
        "최대 오차": round(diff.max(), 6),
        "평균 오차": round(diff.mean(), 6),
        "상관계수": round(np.corrcoef(df[col], calc[col])[0, 1], 6),
    })

result = pd.DataFrame(rows)
print("=== 파생 변수 계산식 검증 결과 ===")
print(result.to_string(index=False))

# ---------------------------------------------------------------
# 5. 일치하지 않는 행이 있으면 예시 출력
# ---------------------------------------------------------------
for col in calc.columns:
    diff = (df[col] - calc[col]).abs()
    bad = diff > TOL
    if bad.any():
        print(f"\n[{col}] 불일치 {bad.sum()}행 예시")
        print(pd.DataFrame({"원본": df.loc[bad, col],
                            "계산값": calc.loc[bad, col].round(4),
                            "오차": diff[bad].round(4)}).head())

# ---------------------------------------------------------------
# 6. 강종 더미 변수 중복 여부 확인
# ---------------------------------------------------------------
steel_sum = df["TypeOfSteel_A300"] + df["TypeOfSteel_A400"]
print("\n=== 강종 더미 변수 확인 ===")
print("A300 + A400 값 분포:", steel_sum.value_counts().to_dict())
if (steel_sum == 1).all():
    print("→ 모든 행에서 합이 1: 두 열은 완전히 겹치는 정보이므로 하나만 사용")

# ---------------------------------------------------------------
# 7. 면적 관련 변수 간 상관 (SigmoidOfAreas는 식 미확인, 상관만 확인)
# ---------------------------------------------------------------
print("\n=== 면적 관련 변수 상관계수 (Spearman) ===")
print(df[["Pixels_Areas", "LogOfAreas", "SigmoidOfAreas"]]
      .corr(method="spearman").round(4))

## 면적 관련 변수인 Pixels_Areas, LogOfAreas, SigmoidOfAreas는 높은 상관관계를 보이지만, 각각 원본 면적과 서로 다른 변환 정보를 나타낸다. 
## 따라서 사전에 변수를 제거하기보다는 세 변수를 모두 유지한 상태에서 분석을 진행하고, 
## 변수 제거 전후의 분류 성능을 비교하여 중복 정보가 모델에 미치는 영향을 확인하고자 한다. 
## 이를 통해 변수 간 상관관계뿐 아니라 실제 예측 성능을 고려해 최종 변수를 선정한다.

=== 파생 변수 계산식 검증 결과 ===
                  변수  일치 행 수  전체 행 수  일치율(%)    최대 오차    평균 오차  상관계수
         Edges_Index    1941    1941   100.0 0.000055 0.000024   1.0
         Empty_Index    1941    1941   100.0 0.000055 0.000024   1.0
        Square_Index    1941    1941   100.0 0.000055 0.000022   1.0
     Outside_X_Index    1941    1941   100.0 0.000055 0.000025   1.0
       Edges_X_Index    1941    1941   100.0 0.000055 0.000020   1.0
       Edges_Y_Index    1941    1941   100.0 0.000055 0.000014   1.0
Outside_Global_Index    1941    1941   100.0 0.000000 0.000000   1.0
          LogOfAreas    1941    1941   100.0 0.000055 0.000025   1.0
         Log_X_Index    1941    1941   100.0 0.000054 0.000025   1.0
         Log_Y_Index    1941    1941   100.0 0.000054 0.000025   1.0
   Orientation_Index    1941    1941   100.0 0.000055 0.000022   1.0
    Luminosity_Index    1941    1941   100.0 0.000055 0.000026   1.0

=== 강종 더미 변수 확인 ===
A300 + A400 값 분포: {1: 1941}
→ 모든 행에서 합이 1: 두 열은 완전히 겹치는 정보

In [9]:
# 1. 결함 유형을 하나의 열로 통합
fault_cols = [
    "Pastry", "Z_Scratch", "K_Scatch", "Stains",
    "Dirtiness", "Bumps", "Other_Faults"
]

df1 = df.copy()

df1["Fault"] = df1[fault_cols].idxmax(axis=1)

# 기존 결함 유형 열 제거
df1 = df1.drop(columns=fault_cols)


# 2. 강종을 하나의 열로 통합
# TypeOfSteel_A300: 1이면 A300, 0이면 A400
df1["TypeOfSteel"] = df1["TypeOfSteel_A300"]

# 기존 강종 열 제거
df1 = df1.drop(columns=["TypeOfSteel_A300", "TypeOfSteel_A400"])


# 3. df1 확인
display(df1.head())
print("df1 크기:", df1.shape)


# 4. df1에서 파생변수 제거하여 df2 생성
derived_cols = [
    "Edges_Index",
    "Empty_Index",
    "Square_Index",
    "Outside_X_Index",
    "Edges_X_Index",
    "Edges_Y_Index",
    "Outside_Global_Index",
    "LogOfAreas",
    "Log_X_Index",
    "Log_Y_Index",
    "Orientation_Index",
    "Luminosity_Index"
]

df2 = df1.drop(columns=derived_cols).copy()


# 5. df2 확인
display(df2.head())
print("df2 크기:", df2.shape)

,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,...,Edges_Y_Index,Outside_Global_Index,LogOfAreas,Log_X_Index,Log_Y_Index,Orientation_Index,Luminosity_Index,SigmoidOfAreas,TypeOfSteel,Fault
0,42,50,270900,270944,267,17,44,24220,76,108,...,1.0000,1.0,2.4265,0.9031,1.6435,0.8182,-0.2913,0.5822,1,Pastry
1,645,651,2538079,2538108,108,10,30,11397,84,123,...,0.9667,1.0,2.0334,0.7782,1.4624,0.7931,-0.1756,0.2984,1,Pastry
2,829,835,1553913,1553931,71,8,19,7972,99,125,...,0.9474,1.0,1.8513,0.7782,1.2553,0.6667,-0.1228,0.2150,1,Pastry
3,853,860,369370,369415,176,13,45,18996,99,126,...,1.0000,1.0,2.2455,0.8451,1.6532,0.8444,-0.1568,0.5212,0,Pastry
4,1289,1306,498078,498335,2409,60,260,246930,37,126,...,0.9885,1.0,3.3818,1.2305,2.4099,0.9338,-0.1992,1.0000,0,Pastry


df1 크기: (1941, 27)


,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,Length_of_Conveyer,Steel_Plate_Thickness,SigmoidOfAreas,TypeOfSteel,Fault
0,42,50,270900,270944,267,17,44,24220,76,108,1687,80,0.5822,1,Pastry
1,645,651,2538079,2538108,108,10,30,11397,84,123,1687,80,0.2984,1,Pastry
2,829,835,1553913,1553931,71,8,19,7972,99,125,1623,100,0.2150,1,Pastry
3,853,860,369370,369415,176,13,45,18996,99,126,1353,290,0.5212,0,Pastry
4,1289,1306,498078,498335,2409,60,260,246930,37,126,1353,185,1.0000,0,Pastry


df2 크기: (1941, 15)
